In [ ]:
#| hide
from 2023_easy_pin_detection.core import *

This file will become your README and also the index of your documentation.

## Install

```sh
pip install 2023_easy_pin_detection
```

# Model History

- ``ModelV10`` -> It is one image model, with prediction threshold `0.9`
- ``ETPD-WAR1_00.onnx`` -> Same model like previous but threshold for model prediction is 0.5 instead of 0.9
- ``ETPD-WAR1_01.onnx`` -> Same model like `ETPD-WAR1_00.onnx` but image1 is binaraized before passing to model and threshold for `image1` is `0.5`
- ``ETPD-WAR1_02.onnx`` -> Same model like `ETPD-WAR1_00.onnx` but image1 is binaraized before passing to model and threshold for `image1` is `0.3` and also smaller object is removed, because for becuase of edge, there are some smaller object found after multiplication
- ``ETPD-WAR1_03.onnx`` -> Lots of Missing pin found. However the pin looks like there is some black steps in the pin found. So new model Trained and found to solve the black strips in the pin.
- ``ETPD-WAR1_04.onnx`` -> Again found some missing pin. But the issue there is some change in the moudle color, so currently createing huge problem. So new model trained with new color and found to solve
- ETPD-WAR1_05.onnx -> because of change of module and pin type, lots of image failed. pins are black
                IoU-> 95.18-(one image and two image model saved in ModelV12)

#  Problem history

- 17.09.2024
    - from 14-09 to 17-09 all failed images analayzed. Goal is to find whehter Modelv5 is passed or not. As new version is trained mostly black pins and current model has sometimes issue with black pins. 
-- 18.07.2024
    - lot 81864464 (ET14 failed in Tableau-10)
-- 18.08.2024
    - lot 81867539 
    - plus some dirty pins so showing missing pins
    - saved data `/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/`
    - Trained with single pin missing data

-- There is some additional pin available. The problem occurs, when actual pin mask has two objects found ( means, pin is detected but some small area still available)
    - Current procedure is only for reflection but not for smaller addtional object in the pin area.  
    - Normally we have minmum pixel area is selected, this should remove the problem

-- 03.09.2024
  - Saved all missing pin in a folder, to create test data.
  - Made sure so that, last 4 week Missing data in a separate folder (because different module color and pin )
  - Training res-attention Unet with current data

-- 04.09.2024
  - Failed 0209 --v5 
    -- missing - 2
    -- additional  - 8 
    -- pass -162 (actully) - 2 found small problatic

-- 05.09.2024
  - Fail 0209 Patch image 256 with residual attention UNet
    - Mssing - 2
    - But lotts of additonal and some pin shape is not correct. 
    - Solution will be train with those pin and retrain, but it is then again work, so will not done now.
    - Also other resoltion need to check (64, 128, 512)

  - Failed 3108 --v5
    - from 27 images all passed


  - Failed 3008 --v5
    - from 249 images no missing pin
    - 2 images found in addtional pin filed missing pin actually
  - Trained with new data
  - Failed 2908 --v5
    - from 287 images no missing pin
    - 5 images found in addtional pin,because additional pin , so not considered here
  - Failed 2808 --v5
    - from 163 images 1 missing pin, which is found in addtional pin folder

  - After that Failed upto 1408 
    - no failed images found, I didn't check the folder, log says no failed imags

## How to use

## Training

- In case of Multi GPU training, use `training/training_with_config_multi_gpu.sh` script 
  - `CONFIG_PATH` - path to the config file
  - Inside training there is some other mutli gpu training script, but I need to check them, whehter both are same or not


# Create masks and create overlay and copy pass fail image


Script name = `mask_generation.sh`
- TWO_IMAGE=yes or no
-IMAGE_SOURCE=
-IMAGE1_SOURCE=
-PR_IM_PATH

### Crop image
- `crop_image.sh`
  - `IM_PATH` - image path
  - `IM_WIDTH` - expected image width after crop
  - `IM_HEIGHT` - expected image height after crop
  - `IM_SAVE_PATH` - save path
  - `--c_crop` - crop image from center if it is commented out, then it will be cropped like in vision pc

## fit rotated rectangle to masks
- `preprocessing/zero_degree_solder_pin/fit_rotated_rectangle_to_masks.sh`
- `preprocessing/zero_degree_solder_pin/fit_rotated_rectangle_to_masks.py`
## Extract single pins from masks
   - `preprocessing/zero_degree_solder_pin/extract_pins.sh`

# Generate mask from two images 
- `mask_generation_two_image.sh`
- before running this script, make sure you unzipped the image
    - `vi_pc_unzip.sh`


### Create overlay mask

- To create a overlay mask from image and mask folder, use `overlay_mask.sh`
  - `IM_PATH`- where images are saved
  - `MASK_PATH`- where masks are saved
  - `OVERLAY_SAVE_PATH`- where to save the overlay masks


### Correct mask

- Suppose there are some masks and you want to correct those masks. What does correction means
  - Remove smaller objects
  - fill holes inside the objects
  - apply rectangle mask to the objects
- Use `process_mask.sh` script
  - `MASK_DIR`  - where masks are saved
  - `OUTPUT_DIR` - where corrected masks will be saved
  - `REMOVE_OBJECT_SIZE` -90000 will be default

### Correct mask based on recipe wise pin position

- We know that, for specific recipe will have a specific pin position.
- pre-requisite:
    - the recipe map is already created in pin_map.py script
- Script can be used `recipewise_pin_position.sh`
    - `MSK_PATH`
    - `IM_PATH`
    - `RECIPE_NAME`
    - `CORRECTED_MASK_PATH` - if default is used, then in same folder named as processed_masks folder will be created and masks will be created here
- After this script new masks will be created, additionally, `overlay_masks` folder will be created and overlay of mask and image will be created here.

### Extract problem pins from processed image and save those single pin in a folder

- why, 
    - so that we can generate new image with same part or pin in a single image
    - script name is `extract_problem_pin.sh`
      - `IM_PATH`- image path
      - `MSK_PATH`- mask path
      - `PR_IM_PATH` - problem image path for processed image 
      - `SAVE_IM_PATH` - save image path (whether to save image path or not)
      - `SAVE_MSK_PATH` - save mask path (whether to save mask path or not)

### Extract addtional pins

- Suppose you have some additional pins but you have processed image is not aviable, as you are checking it manually. Then use
> `extract_smaller_mask.sh`
- IM_PATH - is not the single image path, but it will look all the images inside this folder
- MASK_PATH - same like IM_PATH
- MASK_SIZE - is the size of the mask, which is the size of the image
- TMP_PATH - is the tempalte Path, make sure you have a envronemnt varialbe TEMP_PATH is exist for templates path
- Inside `12_extract_pins_where_msk_area_small.ipynb` there are some other funcktionlity available to save the image and mask for single pin

## Extract pins from whole image

> Lets say you have an image and mask. And you want to create single pin image and mask from the whole image based on mask. As you know from the mask where are the pins, you just crop those parts from the image and mask and save them. 
- Goal will be to generate new image with those singel images and masks

- ``extract_patches.sh``

### Generate Image

- We need one full image
- One addtional pin image or missing pin image
- in case of mask is available (additional pin needs an ideal mask)
- script will be `generate_image.sh`
  - `IM_PATH`- full image path
  - `MSK_PATH`- ideal mask path (in case of additional pin)
  - `ADD_PIN_PATH`- additional pin path (small or single pin or pin like single image part)
  - `TEMPLATE_PATH`- template path for specific recipe
  - `GEN_IM_SAVE_PATH` - generated image save path
  - `GEN_MSK_SAVE_PATH` - generated mask save path

## Noise Image and mask saving

- Saving noised, sharpened or different lighting function and blurred images
  - `./noise_sharpen_gen.sh`
  - in case of different lighting,
    -- lighting
  - in case of blurred images,
    -- blur
  - in case of sharpened images,
    -- sharpen

# Multiplication with Image1

- In case of threshold 0.5 for image1 seems problmatic. Because after 0.5 image1 pin lost its rectangle shape 
- But there will be some addtional pin. We need to remove smaller pin
- Threshold 0.3 seems to good for shape construction and additional pin removal
    - Copying all images with addtional pin and then will be checked out whether any addtional pin found or not
           -  
    - Also Missing pin will also be checked out for this. 
            - check failed images saved in `\home\ai_sintercra.work\Fail_investigation\Missing_lead\missing_all\Failed_images\Failed_images_unzip\main_im2_cropped_masks\threshold5\failed\missing\overlay`

# Create a Two image model from a One image model

`model_eval.two_image_model.sh`

### Create Three channel dataset from zip file

`vi_pc_unzip.sh`
- `P_DIR` - inside this directory each folder will be looked up. The zip file should be situated inside subfolder.
    - e.g. `P_DIR=/mnt/test/` the zip file should be inside `/mnt/test/sub_test/` folder

### Generate three channel image, if you have `im1` and `im2`

`create_single_image_from_im1_im2.sh`
- `IM1_PATH` - image 1 path
- `IM2_PATH` -image 2 path
- `IM_SAVE_PATH` - image save path
- `SECOND_CHANNEL` - `im1` or `im2`


# In case of we want to create a patch model

> So in case we want to patch images in smaller part

## Training patch model

In [ ]:
- `training_with_config.sh` -> need to change config file 

- e.g. for patch `512` the config file should be `patch_image_512.yml`
  - in config file make sure input size is 256
  - make sure input training and validation mask and image in right place
  - make sure image height and image width is correct
  - make sure saving model path is correct, pretrained True or False
  - input_shape is correct

## Patching image with only minimum pixel mask available

In [ ]:
- `patching_image.sh` or `patch_image_256.sh` -> both could be used
  - make sure input image path is correct
  - make sure output image path is correct
  - make sure patch size is correct